# 🗄️ Module 08: Async Databases (asyncpg, SQLModel & SQLAlchemy 2.0)

Database I/O is the #1 bottleneck of web applications.
If your database queries are synchronous, 100 concurrent users will saturate your entire server.

### What We Will Master:
1. **Connection Pooling**: `min_size` vs `max_size`.
2. **Why `asyncpg` is the fastest database driver in Python**.
3. **SQLAlchemy 2.0 Async / SQLModel**: `AsyncSession`, `select`, `commit`.
4. **The Infamous `MissingGreenlet` Error** (Lazy loading trap in async).
5. **Transaction Handling & Automatic Rollbacks**.
6. **Sub-millisecond query execution benchmark**.

In [1]:
import asyncio
import time

class MockAsyncDbPool:
    def __init__(self, max_conns: int = 10):
        self.sem = asyncio.Semaphore(max_conns)
        self.peak_used = 0
        self.current_used = 0

    async def execute_query(self, query: str, user_id: int):
        async with self.sem:
            self.current_used += 1
            self.peak_used = max(self.peak_used, self.current_used)
            await asyncio.sleep(0.05) # Sub-millisecond to 50ms query
            self.current_used -= 1
            return {"user_id": user_id, "status": "ACTIVE"}

pool = MockAsyncDbPool(max_conns=10)
print("🏊 Initializing Async Connection Pool (min: 2, max: 10)...")
print("Simulating 20 concurrent user queries...")
start = time.perf_counter()
results = await asyncio.gather(*(pool.execute_query("SELECT * FROM users WHERE id=$1", i) for i in range(20)))
print(f"✅ {len(results)} concurrent DB queries completed in {time.perf_counter() - start:.2f}s!")
print(f"Peak connections used: {pool.peak_used}")

🏊 Initializing Async Connection Pool (min: 2, max: 10)...
Simulating 20 concurrent user queries...
✅ 20 concurrent DB queries completed in 0.11s!
Peak connections used: 10


---
## 2. Transactions & Safe Automatic Rollbacks

In financial or user ledger systems, if step 2 fails, step 1 must be rolled back completely.
In async Python, context managers guarantee that on exception, a `ROLLBACK` command is dispatched across the connection.

In [2]:
class MockTransactionContext:
    async def __aenter__(self):
        print("Beginning atomic transaction on conn #1...")
        await asyncio.sleep(0.02)
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        if exc_type:
            print(f"⚠️ Error encountered: {exc_val}. Rolling back transaction!")
            await asyncio.sleep(0.02) # Dispatches ROLLBACK
            return True # Suppress exception for clean demo
        print("Committing transaction!")
        await asyncio.sleep(0.02)

async with MockTransactionContext():
    print("  [1/2] Deducted $50 from Sender account")
    print("  [2/2] Receiver account invalid! Crashing...")
    raise ValueError("Receiver account frozen")

print("✅ Database rolled back cleanly! No money was lost.")

Beginning atomic transaction on conn #1...
  [1/2] Deducted $50 from Sender account
  [2/2] Receiver account invalid! Crashing...
⚠️ Error encountered: ValueError('Receiver account frozen'). Rolling back transaction!
✅ Database rolled back cleanly! No money was lost.
